# 两个 100-round DP-FedAvg 结果的 reconciliation

对 GEF2017、AusGrid2013 已完成的 100-round 训练结果，计算 **base、mints、wlsv、ols**。

- 使用各自 100-round 模型保存的训练残差，直接复用原 `FHF_Cloud` 实现，`ridge=1e-6`；不重训、不额外添加预测或残差噪声。
- mints = `mint_shrinkage`，wlsv = `mint_var`，ols = `mint_ols`。
- 测试真值只用于评分；MASE / RMSSE 沿用训练目标 lag-1 差分分母，先逐序列计算再取算术平均。
- 参数从各自原始 privacy.json 读取，并验证 100 轮与 100 次聚合。
- 输入：`DP_fhf/outputs/01_dp_fedavg_100round/<dataset>/`。
- 输出：**`DP_fhf/outputs/dp_100round_reconciliation/`**。

`training_epsilon` / `training_delta` 只记录原 DP-FedAvg 训练隐私参数，不代表本次使用未额外私有化训练残差的整体隐私保证。

In [ ]:
# Locate the project package from a notebook folder or the project root.
from pathlib import Path
import os
import sys
_pphf_start = Path.cwd().resolve()
_pphf_override = os.environ.get("PPHF_PROJECT_ROOT") or os.environ.get("DP_FHF_PROJECT_ROOT")
_pphf_candidates = ([Path(_pphf_override).expanduser()] if _pphf_override else []) + [_pphf_start, *_pphf_start.parents]
PROJECT_ROOT = next((p.resolve() for p in _pphf_candidates if (p / "PPHF_modules" / "fhf" / "FHF_runner.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Cannot locate PPHF_modules. Set PPHF_PROJECT_ROOT or start Jupyter inside the project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pathlib import Path
import os
import sys
import importlib
import pandas as pd
from IPython.display import display

# 在项目任意子目录启动；也可通过环境变量指定项目根目录。
candidates = []
if os.environ.get("DP_FHF_PROJECT_ROOT"):
    candidates.append(Path(os.environ["DP_FHF_PROJECT_ROOT"]).expanduser())
candidates.extend([Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in candidates
             if (p / "PPHF_modules" / "fhf").is_dir() and (p / "DP_fhf").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Set DP_FHF_PROJECT_ROOT to the FHF_NDSH project directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from PPHF_modules.dp_fhf import reconciliation_100round
importlib.reload(reconciliation_100round)

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
print("Project:", ROOT)
print("Python:", sys.executable)

In [ ]:
DATASETS = ("GEF2017", "AusGrid2013")
OUTPUT_ROOT = ROOT / "DP_fhf" / "outputs" / "dp_100round_reconciliation"
RIDGE = 1e-6
SAVE_FORECASTS = True
print("Output:", OUTPUT_ROOT)

## 计算与保存

重新计算两组结果，验证原始 base 指标、矩阵投影与层级加总一致性。完整批次成功后才替换输出目录。

In [ ]:
summary = reconciliation_100round.run_100round_reconciliation(
    root=ROOT, datasets=DATASETS, output_root=OUTPUT_ROOT,
    ridge=RIDGE, save_forecasts=SAVE_FORECASTS,
)
assert len(summary) == 4 * len(DATASETS)
display(summary[["dataset", "num_rounds", "noise_multiplier", "clip_norm", "method", "MAE", "RMSE", "MASE", "RMSSE"]])

## MASE / RMSSE 并排比较

In [ ]:
comparison = summary.pivot(index="dataset", columns="method", values=["MASE", "RMSSE"])
comparison = comparison.reindex(columns=pd.MultiIndex.from_product(
    [["MASE", "RMSSE"], ["base", "mints", "wlsv", "ols"]], names=comparison.columns.names))
display(comparison)
print("Relative change vs base; negative means lower error:")
display(summary.loc[summary.method != "base", ["dataset", "method", "change_MASE_pct_vs_base", "change_RMSSE_pct_vs_base"]])

## 文件与验证

根目录包含 `metrics_summary.csv`（8 行）、`metrics_wide.csv`（2 行）、逐序列指标及矩阵检查。
每个数据集文件夹保存逐点预测 `reconciled_forecasts.csv.gz`、S/G/P/W 矩阵、该组指标及输入/代码 SHA256。
压缩预测可直接通过 `pd.read_csv` 读取。

In [ ]:
import json
manifest = json.loads((OUTPUT_ROOT / "manifest.json").read_text())
checks = pd.read_csv(OUTPUT_ROOT / "matrix_diagnostics.csv")
assert manifest["status"] == "complete"
assert manifest["runs_completed"] == len(DATASETS)
assert checks.PS_minus_S_max_abs.max() < 1e-7
assert checks.coherence_relative.max() < 1e-9
assert all(run["source_files_unchanged"] for run in manifest["run_manifests"])
print("Maximum |PS-S|:", checks.PS_minus_S_max_abs.max())
print("Maximum relative coherence error:", checks.coherence_relative.max())
print("Summary:", OUTPUT_ROOT / "metrics_summary.csv")
print("Wide comparison:", OUTPUT_ROOT / "metrics_wide.csv")